# 02 — Preprocessing
Turn cleaned reviews into a binary label and a train/test split.
Input:  raw_data/processed/amazon_clean.csv   (from notebook 1: text + raw stars)
Output: raw_data/processed/train.csv, test.csv (text + label)
No model here — that's notebook 3.

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import pandas as pd
from sklearn.model_selection import train_test_split
from bparadigm.paths import AMAZON_CLEAN, TRAIN_CSV, TEST_CSV, PROCESSED

In [3]:
df = pd.read_csv(AMAZON_CLEAN)
print(df.shape, "| exists:", (AMAZON_CLEAN).exists())
df.head()

(20406, 2) | exists: True


,text,stars
0,"I registered on the website, tried to order a ...",1.0
1,Had multiple orders one turned up and driver h...,1.0
2,I informed these reprobates that I WOULD NOT B...,1.0
3,I have bought from Amazon before and no proble...,1.0
4,If I could give a lower rate I would! I cancel...,1.0


In [4]:
df["stars"].value_counts(dropna=False).sort_index()

stars
1.0    12953
2.0     1203
3.0      825
4.0     1193
5.0     4232
Name: count, dtype: int64

In [5]:
def stars_to_label(s):
    if s in (1, 2): return 0    # Negative -> 3-star dropped due to ambiguity
    if s in (4, 5): return 1    # Positive
    return None

df["label"] = df["stars"].map(stars_to_label)
df["label"].value_counts(dropna=False)    # NaN -> dropped

label
0.0    14156
1.0     5425
NaN      825
Name: count, dtype: int64

In [6]:
before = len(df)
df = df[df["label"].notna()].copy()
df["label"] = df["label"].astype(int)
print(f"dropped {before - len(df)} unlabelled rows")
df.shape

dropped 825 unlabelled rows


(19581, 3)

In [7]:
train_df, test_df = train_test_split(
    df[["text", "label"]],
    test_size=0.2,
    stratify=df["label"],   # the load-bearing argument
    random_state=42,        # convention, just memorise: fixes the shuffle so runs reproduce
)
print("train:", train_df.shape, "| test:", test_df.shape)

train: (15664, 2) | test: (3917, 2)


In [8]:
PROCESSED.mkdir(parents=True, exist_ok=True)
train_df.to_csv(TRAIN_CSV, index=False)
test_df.to_csv(TEST_CSV, index=False)
print("saved:", (TRAIN_CSV).exists(), (TEST_CSV).exists())

saved: True True
